# 23 — View / state figure-choice experiments

Thin notebook: it only **imports** and **calls** `src/view_state_experiments.py`, then **displays** the results. It reads labels and slices notebook 22's embeddings. It opens no image, runs no model and samples nothing.

**Question:** which figure should represent an aircraft in the embedding space? All four rules cover **every aircraft that has a G1 code**:
* `exp1_view_first`: the best view, then the best flight state.
* `exp2_state_first`: the best state, then the best view.
* `exp3_main`: the figure the labeller marked as main. An aircraft whose main is not a whole-vehicle figure takes exp1's pick, and this is flagged.
* `exp4_slots_mean`: one figure per view slot (Perspective, Plan, Side), averaged over the slots the aircraft has.

**Side tests:** `exp4_slots_2` and `exp4_slots_3` concatenate the slot vectors. They cover only the aircraft that have every slot.

**Unit:** the aircraft (`<patent>_ua<N>`). G1 and the main marker are stored per aircraft.
**Input:** Stage 04's tables, passed through notebook 20's gates, and `embeddings/dinov2-large_518/` from notebook 22.
**Output:** `<paths.pipeline_root>/view_state_experiments/`, which holds the figure table, the manifests, `aircraft_common.csv`, `VIEW_STATE_REPORT.md` and `exp_embeddings/<exp>/`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config_loader import load_config
from src import view_state_experiments as vse

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 70)
cfg = load_config()
OUT = vse.out_dir(cfg)
print('output          :', OUT)
print('canonical state :', vse.CANONICAL_STATE)
print('primary matrix  :', vse.EMB_TAG, f'layer {vse.PRIMARY_LAYER}', vse.PRIMARY_POOLING)

output          : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/view_state_experiments
canonical state : Cruise
primary matrix  : dinov2-large_518 layer 22 cls


## Step 1: figure table
Notebook 20's gates are applied in order: approval, domain gate, D1/D2. The last gate, scope, is kept here as a column so the number of part figures it drops is visible. Every later step uses whole-vehicle figures only. The whole-vehicle set must equal `processed/518`, the set notebook 22 embedded.

In [2]:
master, cand, funnel = vse.build_candidates(cfg)
t = vse.add_derived(vse.build_figure_table(cand, cfg))
t.to_csv(OUT / 'figure_table.csv', index=False)
w = vse.whole_vehicle(t)
scope_dropped = int((t['scope'] != 'whole_vehicle').sum())
n_no_fig_number = int(w['fig_number'].isna().sum())

proc = pd.read_csv(Path(cfg['paths']['pipeline_root']) / 'processed' / '518' / 'manifest.csv')
assert set(w['fig_id']) == set(proc['figure_uid']), 'whole-vehicle figures differ from processed/518: rerun 20, 21, 22'

display(funnel)
print(f"{len(t)} figures after the gates; the scope gate drops {scope_dropped} part figures "
      f"-> {len(w)} whole-vehicle figures of {w['aircraft_id'].nunique()} aircraft "
      f"({w['patent_id'].nunique()} patents)")
print('figures with no FIG number (_Fu crops):', n_no_fig_number)
display(t['scope'].value_counts())
w[['aircraft_id', 'fig_id', 'scope', 'view8', 'state_raw', 'g1_code', 'is_main', 'fig_number']].head()

,step,removed,remaining,aircraft_remaining
0,figures on file,0,5855,NaN
1,figure not approved,3940,1915,841.0
2,image file missing,0,1915,841.0
3,patent not approved,2,1913,840.0
4,no aircraft row,0,1913,840.0
5,aircraft not approved,0,1913,840.0
6,domain gate: UAVSimilar,184,1729,754.0
7,domain gate: ElectricSimilar,66,1663,719.0
8,domain gate: STOLSimilar,3,1660,718.0
9,"duplicate patent (D1/D2, points at its original)",74,1586,677.0


1585 figures after the gates; the scope gate drops 18 part figures -> 1567 whole-vehicle figures of 677 aircraft (585 patents)
figures with no FIG number (_Fu crops): 472


scope
whole_vehicle    1567
part               18
Name: count, dtype: int64

,aircraft_id,fig_id,scope,view8,state_raw,g1_code,is_main,fig_number
0,AT503689A1_ua1,AT503689A1_ua1/AT503689A1_fig_01_crop_0_F2.png,whole_vehicle,Side,Hover,CVT,False,2.0
1,AT503689A1_ua1,AT503689A1_ua1/AT503689A1_fig_01_crop_1_Fu.png,whole_vehicle,Back,Cruise,CVT,True,NaN
2,AU2020100605A4_ua1,AU2020100605A4_ua1/AU2020100605A4_fig_03_crop_0_F2.png,whole_vehicle,Front-Isometric,Cruise,CVT,True,2.0
3,AU2020100605A4_ua1,AU2020100605A4_ua1/AU2020100605A4_fig_04_crop_0_F3.png,whole_vehicle,Side,Transition,CVT,False,3.0
4,AU2020100605A4_ua1,AU2020100605A4_ua1/AU2020100605A4_fig_07_crop_0_F4c.png,whole_vehicle,Rear-Isometric,Cruise,CVT,False,4.0


## Step 2: view4 and state4
* **view4:** Top and Bottom/Down map to Plan; Front and Back to FrontRear; Side to Side; Front-Isometric, Rear-Isometric and Generic 3D to Perspective.
* **state4, invariant G1 codes** (RC, MR, SLC, HB, PFV, TB, PTC): always `Invariant`, whatever the label says.
* **state4, variant codes** (TW, TR, DS, CVT, SRW): Hover and Cruise stay as they are. `Invariant` (the drawing fits both states) becomes `Both`. A blank label becomes `Missing`. Transition and Other become `Other`.
* **State order when a rule picks a figure:** canonical > Both > the other real state > Other > Missing.
* SRW is variant, as in the wizard, which asks the state on every SRW figure (ruling 2026-09-18). The brief's TP is the wizard's TR.
* The two aircraft with no G1 (the G1 "unclassifiable" override) are shown as `NoG1`.

In [3]:
ct_g1 = pd.crosstab(w['g1_code'].replace('', '(none)'), w['state4'], margins=True)
ct_view = pd.crosstab(w['view4'], w['state4'], margins=True)
display(ct_g1, ct_view)

state4,Both,Cruise,Hover,Invariant,Missing,NoG1,Other,All
g1_code,,,,,,,,
(none),0,0,0,0,0,4,0,4
CVT,4,123,132,0,10,0,19,288
DS,1,8,4,0,0,0,2,15
HB,0,0,0,34,0,0,0,34
MR,0,0,0,107,0,0,0,107
PFV,0,0,0,23,0,0,0,23
PTC,0,0,0,41,0,0,0,41
RC,0,0,0,22,0,0,0,22
SLC,0,0,0,350,0,0,0,350


state4,Both,Cruise,Hover,Invariant,Missing,NoG1,Other,All
view4,,,,,,,,
FrontRear,2,46,28,60,2,0,6,144
Perspective,12,229,242,309,5,2,39,838
Plan,3,65,74,168,5,1,8,324
Side,2,60,69,105,1,1,23,261
All,19,400,413,642,13,4,76,1567


## Step 3: coverage
The canonical state is whichever of Hover and Cruise has more variant aircraft with at least one Perspective figure in that state. The module constant `CANONICAL_STATE` must agree with the data.

In [4]:
lost = vse.aircraft_without_whole_vehicle(vse.domain_aircraft(master, cfg), t)
stats = vse.coverage_stats(w, lost)
canonical = vse.decide_canonical(stats)
assert canonical == vse.CANONICAL_STATE, f'the data now favour {canonical}: update CANONICAL_STATE'
print(json.dumps(stats, indent=1))

{
 "aircraft": 677,
 "figures": 1567,
 "fig_per_aircraft": {
  "min": 1,
  "median": 2.0,
  "max": 10,
  "distribution": {
   "1": 201,
   "2": 253,
   "3": 106,
   "4": 75,
   "5": 24,
   "6": 10,
   "7": 4,
   "8": 3,
   "10": 1
  }
 },
 "variant_aircraft": 345,
 "variant_persp_hover": 209,
 "variant_persp_cruise": 215,
 "variant_persp_both": 166,
 "variant_persp_neither": 87,
 "slot2_aircraft": 132,
 "slot2_pct": 19.5,
 "slot3_aircraft": 50,
 "slot3_pct": 7.4,
 "no_whole_vehicle_figure": []
}


## Step 4: selection manifests
Aircraft with no G1 code are left out of every manifest because they have no class. exp3 lists the aircraft with no single whole-vehicle main in `exp3_conflicts.csv`; those aircraft take exp1's pick. exp4 gives an aircraft with no Perspective, Plan or Side figure its best FrontRear figure. The comparison set, `aircraft_common.csv`, is therefore every aircraft with a G1 code.

In [5]:
excluded = vse.excluded_no_g1(w, master)
ws = w[w['g1_group'] != 'none']
exp1 = vse.select_single(ws, 'view')
exp2 = vse.select_single(ws, 'state')
exp3, exp3_conflicts = vse.select_main(ws, t, fallback=exp1)
exp4 = vse.select_slots(ws)
common = vse.common_aircraft(exp1, exp2, exp3, exp4)
y_common = vse.ground_truth(common, ws)

files = {
    'exp1_view_first.csv': exp1,
    'exp2_state_first.csv': exp2,
    'exp3_main.csv': exp3,
    'exp3_conflicts.csv': exp3_conflicts,
    'exp4_slots.csv': exp4,
    'aircraft_common.csv': y_common,
    'excluded_aircraft.csv': excluded,
}
for name, df in files.items():
    df.to_csv(OUT / name, index=False)
sizes = {k: len(v) for k, v in files.items()}
display(pd.Series(sizes, name='rows'), excluded, exp3_conflicts)
exp1.head()

exp1_view_first.csv      675
exp2_state_first.csv     675
exp3_main.csv            675
exp3_conflicts.csv         2
exp4_slots.csv           973
aircraft_common.csv      675
excluded_aircraft.csv      2
Name: rows, dtype: int64

,aircraft_id,patent_id,reason
0,CN109263914A_ua1,CN109263914A,no G1 topType (G1 quick override; note: Convertiplane)
1,CN109263949A_ua1,CN109263949A,"no G1 topType (G1 quick override; note: strange form, residual one..."


,aircraft_id,patent_id,n_main,reason,main_fig_ids
0,US2021309351A1_ua1,US2021309351A1,0,main figure is not whole-vehicle,US2021309351A1_ua1/US20210309351A1_D00004_crop_0_F48.png
1,US6367736B1_ua1,US6367736B1,0,main figure is not whole-vehicle,US6367736B1_ua1/US6367736B1_fig_02_crop_0_F2.png


,aircraft_id,patent_id,fig_id,image_path,slot,view4,state4,rule_applied
0,AT503689A1_ua1,AT503689A1,AT503689A1_ua1/AT503689A1_fig_01_crop_0_F2.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & ...,single,Side,Hover,"view-first: view4=Side (rank 2), state4=Hover (rank 2), fig_number..."
1,AU2020100605A4_ua1,AU2020100605A4,AU2020100605A4_ua1/AU2020100605A4_fig_03_crop_0_F2.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & ...,single,Perspective,Cruise,"view-first: view4=Perspective (rank 0), state4=Cruise (rank 0), fi..."
2,AU2020100605A4_ua2,AU2020100605A4,AU2020100605A4_ua2/AU2020100605A4_fig_05_crop_0_F4a.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & ...,single,Perspective,Hover,"view-first: view4=Perspective (rank 0), state4=Hover (rank 2), fig..."
3,AU2020100605A4_ua3,AU2020100605A4,AU2020100605A4_ua3/AU2020100605A4_fig_10_crop_0_F5a.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & ...,single,Perspective,Cruise,"view-first: view4=Perspective (rank 0), state4=Cruise (rank 0), fi..."
4,BR112023017877B1_ua1,BR112023017877B1,BR112023017877B1_ua1/img0001_crop_1_Fu.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & ...,single,Plan,Both,"view-first: view4=Plan (rank 1), state4=Both (rank 1), fig_number=..."


## Step 5: overlap
This is the share of aircraft for which two single-figure rules pick the same `fig_id`, on the whole comparison set.

In [6]:
variant_ids = set(ws.loc[ws['g1_group'] == 'variant', 'aircraft_id'])
ov = vse.overlap({'exp1': exp1, 'exp2': exp2, 'exp3': exp3}, common, variant_ids)
ov

,pair,all_n,all_same_pct,variant_n,variant_same_pct
0,exp1 vs exp2,675,96.6,345,93.3
1,exp1 vs exp3,675,83.6,345,87.0
2,exp2 vs exp3,675,83.6,345,87.0


## Step 6: embedding folders
All folders use one matrix (`EMB_TAG`, layer `PRIMARY_LAYER`, pooling `PRIMARY_POOLING`), sliced by `fig_id`. Every selected figure must already have a row; nothing is extracted here.
* **The four main folders** hold every aircraft of `aircraft_common.csv`, in its row order.
* **The side tests** `exp4_slots_2` and `exp4_slots_3` keep only the aircraft that have every slot, and list the rest in `dropped_ids.csv`.

In [7]:
X, meta, emb = vse.load_primary_matrix(cfg)
miss = vse.missing_vectors({'exp1': exp1, 'exp2': exp2, 'exp3': exp3, 'exp4': exp4}, meta)
assert miss.empty, f'{len(miss)} selected figures have no embedding row:\n{miss}'

EXPS = {  # folder: (manifest, manifest file, slots, mode)
    'exp1_view_first': (exp1, 'exp1_view_first.csv', None, 'concat'),
    'exp2_state_first': (exp2, 'exp2_state_first.csv', None, 'concat'),
    'exp3_main': (exp3, 'exp3_main.csv', None, 'concat'),
    'exp4_slots_mean': (exp4, 'exp4_slots.csv', None, 'mean'),
    'exp4_slots_2': (exp4, 'exp4_slots.csv', vse.SLOTS[:2], 'concat'),   # side test: aircraft with both slots
    'exp4_slots_3': (exp4, 'exp4_slots.csv', vse.SLOTS, 'concat'),       # side test: aircraft with all 3 slots
}
folders, n_slots = {}, {}
for name, (man, fname, slots, mode) in EXPS.items():
    Xo, ids, figs, dropped = vse.build_matrix(man, common, slots, X, meta, mode=mode)
    y = vse.ground_truth(ids, ws)
    m = vse.experiment_meta(name, fname, slots, emb, canonical, Xo, dropped, mode=mode)
    vse.write_experiment(OUT / 'exp_embeddings' / name, Xo, ids, figs, y, m, dropped)
    folders[name] = {'rows': len(ids), 'dim': int(Xo.shape[1]), 'dropped_missing_slot': len(dropped)}
    if mode == 'mean':
        n_slots = {int(k): int(v) for k, v in figs['n_slots'].value_counts().items()}
print('slots per aircraft in exp4_slots_mean:', dict(sorted(n_slots.items())))
pd.DataFrame(folders).T

slots per aircraft in exp4_slots_mean: {1: 427, 2: 198, 3: 50}


,rows,dim,dropped_missing_slot
exp1_view_first,675,1024,0
exp2_state_first,675,1024,0
exp3_main,675,1024,0
exp4_slots_mean,675,1024,0
exp4_slots_2,131,2048,544
exp4_slots_3,50,3072,625


## Step 7: ground truth
Each folder has a `y.csv` with `aircraft_id, patent_id, g1_code, family_id`, in the same order as its `aircraft_ids.npy`. The dataset keeps one patent per family, so `family_id` groups the same rows as `patent_id`. The table below is the class distribution on the comparison set. No model is trained here.

In [8]:
classes = vse.class_distribution(y_common)
display(classes)
print('classes with fewer than 20 aircraft:', classes.loc[classes['below_20'], 'g1_code'].tolist())
for name in EXPS:
    d = OUT / 'exp_embeddings' / name
    ids = __import__('numpy').load(d / 'aircraft_ids.npy')
    assert list(pd.read_csv(d / 'y.csv')['aircraft_id']) == list(ids)

,g1_code,aircraft,group,below_20
0,SLC,185,invariant,False
1,TR,160,variant,False
2,CVT,111,variant,False
3,TW,60,variant,False
4,MR,50,invariant,False
5,TB,28,invariant,False
6,PTC,26,invariant,False
7,HB,20,invariant,False
8,RC,11,invariant,True
9,PFV,10,invariant,True


classes with fewer than 20 aircraft: ['RC', 'PFV', 'SRW', 'DS']


## Report

In [9]:
rep = OUT / 'VIEW_STATE_REPORT.md'
vse.write_report(rep, ov=ov, n_common=len(common), n_slots=n_slots, funnel=funnel,
                 scope_dropped=scope_dropped, n_no_fig_number=n_no_fig_number, stats=stats,
                 canonical=canonical, ct_g1=ct_g1, ct_view=ct_view, sizes=sizes, excluded=excluded,
                 conflicts=exp3_conflicts, emb=emb, classes=classes, folders=folders, cfg=cfg)
print('wrote', rep)

wrote /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/view_state_experiments/VIEW_STATE_REPORT.md
